# HALO — Marketing Assistant for Hairstylists
## Notebook 3: Polished Gradio Frontend Interface
**Course:** ITAI2377 | **Group:** Icarus Squad

**Team Members:** Oyinade Balogun · Francisco Medina Diaz · Rodrigo Sierra · Katherine Stanton

---

### About This Notebook
This notebook contains the branded, stylist-facing frontend for HALO.
It is a fully self-contained notebook — it re-implements the pipeline
functions from Notebook 2 so it can be run independently without
needing Notebook 2 to be open at the same time.

### What This Notebook Does
| Step | Purpose |
|---|---|
| 1 | Install dependencies |
| 2 | Set up API key, imports, and Drive mount |
| 3 | Load ChromaDB and embedding model |
| 4 | Define the HALO pipeline (retrieval + generation + logging) |
| 5 | Build and launch the polished Gradio interface |

### Notebook Structure
| Notebook | Purpose |
|---|---|
| **Notebook 1** | Data collection, preprocessing, ChromaDB storage |
| **Notebook 2** | RAG pipeline, evaluation framework |
| **Notebook 3** | Polished Gradio frontend interface (this notebook) |

### Prerequisites
- Notebook 1 must be fully run and ChromaDB saved to Drive
- Add your free `GEMINI_API_KEY` to Colab Secrets (key icon in left sidebar)
- Get a free key at [aistudio.google.com](https://aistudio.google.com) — no credit card required

### Google Drive Setup for Team Members
If this is your first time running this notebook:
1. Open Google Drive
2. Click **Shared with me** in the left sidebar
3. Right click **HALO_Project** → **Organize** → **Add shortcut to Drive**
4. Place the shortcut in **My Drive** root
5. The notebook will then find the shared folder automatically

### Sharing the Interface with Your Team
When you run Step 5, Gradio generates a public share link valid for 72 hours.
Share this link with your team and professor — they can use HALO from any
device without needing Colab access or an API key.

---
## Step 1 — Install Dependencies

**Cell explanation:** This cell installs all Python libraries needed for the
frontend interface. The key addition compared to a standard setup is `gradio`,
which creates the chat window interface, and `google-generativeai` which
connects to the Gemini API for response generation. Run this cell once at
the start of every new Colab session.

In [1]:
# 1. Install Dependencies
!pip install -q google-genai chromadb sentence-transformers gradio pandas groq openai opentelemetry-api==1.38.0 opentelemetry-sdk==1.38.0 opentelemetry-exporter-otlp-proto-common==1.38.0 opentelemetry-proto==1.38.0

print('All dependencies installed successfully.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 168.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 158.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 142.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 8.5 MB/s eta 0:00:00
All dependencies installed successfully.


---
## Step 2 — Imports, API Key & Drive Mount

**Cell explanation:** This cell imports all required libraries, loads your
Gemini API key securely from Colab Secrets, and mounts Google Drive.

The API key is never written into the notebook code — it is read from
your personal Colab Secrets panel which is invisible to other team members
and not stored in the shared notebook file. Each team member adds their
own free Gemini key using their existing Google account.

**How to add your Gemini API key:**
1. Go to [aistudio.google.com](https://aistudio.google.com) and sign in with your Google account
2. Click **Get API Key** → **Create API key**
3. Copy the key
4. In Colab, click the 🔑 key icon in the left sidebar
5. Click **+ Add new secret**
6. Name: `GEMINI_API_KEY` | Value: paste your key
7. Toggle **Notebook access** to ON

> Free tier allows **1,500 requests per day** — significantly higher than
> Groq's daily token limit. No credit card required.

In [2]:
# 2. Imports, API Key & Drive Mount
import os
import csv
import time
import datetime
import gradio as gr
import pandas as pd

from groq import Groq
from openai import OpenAI
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb

from google.colab import userdata, drive

# Load Gemini API key safely
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except userdata.SecretNotFoundError:
    GEMINI_API_KEY = None

if GEMINI_API_KEY:
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    # List of free tier models to use as fallbacks
    GEMINI_MODELS = ['gemini-2.5-flash', 'gemini-2.0-flash', 'gemini-1.5-flash']
    print(f'Gemini client ready. Models: {GEMINI_MODELS}')
else:
    print('GEMINI_API_KEY not found. Skipping Gemini configuration.')
    gemini_client = None
    GEMINI_MODELS = []

# Load OpenCode Zen API key safely
try:
    OPENCODEZEN_API_KEY = userdata.get('OPENCODE_API_KEY')
except userdata.SecretNotFoundError:
    OPENCODEZEN_API_KEY = None

if OPENCODEZEN_API_KEY:
    client = OpenAI(
        api_key=OPENCODEZEN_API_KEY,
        base_url="https://opencode.ai/zen/v1"
    )
    # Updated primary model based on OpenCode Zen supported list
    GROQ_MODEL = 'gpt-5.4-mini'
    print(f'OpenCode Zen client ready. Model: {GROQ_MODEL}')
else:
    print('OPENCODE_API_KEY not found in secrets.')
    client = None
    GROQ_MODEL = None

# Mount Google Drive
drive.mount('/content/drive')

# Must match BASE_DIR in Notebooks 1 and 2
BASE_DIR = '/content/drive/MyDrive/HALO_Project'
os.makedirs(f'{BASE_DIR}/logs', exist_ok=True)

print(f'Drive mounted. Working directory: {BASE_DIR}')

Gemini client ready. Models: ['gemini-2.5-flash', 'gemini-2.0-flash', 'gemini-1.5-flash']
OPENCODE_API_KEY not found in secrets.
Mounted at /content/drive
Drive mounted. Working directory: /content/drive/MyDrive/HALO_Project


---
## Step 3 — Load ChromaDB & Embedding Model

**Cell explanation:** This cell loads the sentence transformer embedding model
and connects to the ChromaDB vector database saved by Notebook 1.

The embedding model (`all-MiniLM-L6-v2`) must be identical to the one used
in Notebook 1. This is critical — the model converts user queries into vectors
at runtime, and those vectors must be in the same mathematical space as the
chunk vectors stored in ChromaDB. Using a different model would break retrieval.

The model downloads approximately 80MB on first run and uses a local Colab
cache for all subsequent runs in the same session.

**If you see a collection not found error:** Notebook 1 has not been run yet
or the ChromaDB folder is missing from your Drive. Run Notebook 1 first.

In [3]:
# 3. Load ChromaDB & Embedding Model
# Load the sentence transformer model
# This is the same model used in Notebook 1 to encode the knowledge base
print('Loading embedding model (all-MiniLM-L6-v2)...')
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded.')

# Connect to ChromaDB — must exist from Notebook 1
chroma_path   = f'{BASE_DIR}/chromadb'
chroma_client = chromadb.PersistentClient(path=chroma_path)

try:
    collection   = chroma_client.get_collection('halo_knowledge_base')
    total_chunks = collection.count()
    print(f'ChromaDB connected. Total chunks: {total_chunks}')
    if total_chunks < 100:
        print('WARNING: Chunk count is low. Re-run Notebook 1 with more source files.')
    else:
        print('Knowledge base ready. Proceed to Step 4.')
except Exception as e:
    print(f'ERROR: Could not connect to ChromaDB — {e}')
    print('Make sure Notebook 1 has been fully run and saved to Drive.')

Loading embedding model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.
ChromaDB connected. Total chunks: 667
Knowledge base ready. Proceed to Step 4.


---
## Step 4 — HALO Pipeline Functions

**Cell explanation:** This cell defines all the functions that power HALO's
responses. This notebook is self-contained — these functions duplicate those
in Notebook 2 intentionally so Notebook 3 can be run independently without
Notebook 2 needing to be open.

The pipeline has three stages:

**Stage 1 — Retrieval:** `retrieve_chunks()` encodes the user query as a vector
and searches ChromaDB for the most semantically similar knowledge base chunks.
Optional platform and content type filters narrow the search before similarity
matching runs, improving relevance for specific requests.

**Stage 2 — Generation:** `generate_halo_answer()` assembles the retrieved chunks
into a structured prompt with source labels and sends it to the Groq API.
The system prompt gives HALO its persona, scope, and behavioral rules.
Error handling ensures the interface never crashes if the API is unavailable.

**Stage 3 — Logging:** `halo_chat_with_logging()` wraps the full pipeline and
saves every interaction to a persistent CSV on Drive. Feedback ratings from
the interface (Helpful / Not quite) are saved to a separate CSV.

In [4]:
# 4. HALO Pipeline Functions
# HALO system prompt — defines persona, scope, and behavioral rules
# This is identical to the system prompt in Notebook 2 for consistency
HALO_SYSTEM_PROMPT = """
You are HALO, a friendly and knowledgeable marketing assistant built specifically
for independent hairstylists and small salon owners.

Your job is to help hairstylists with five specific tasks:
1. Writing social media captions and content (Instagram, TikTok, Facebook)
2. Planning seasonal and holiday promotional campaigns
3. Explaining marketing metrics in plain, non-technical language
4. Writing client follow-up and rebooking messages
5. Helping stylists communicate what makes them unique

IMPORTANT RULES:
- Base your responses ONLY on the provided context.
- NEVER use the words 'excerpt', 'excerpts', 'knowledge base', 'provided information',
  or any phrase that reveals you are reading from documents. This is non-negotiable.
- Speak entirely as a confident marketing advisor who knows this topic well.
- If you lack specific information on a topic, say something natural like
  'I'd recommend experimenting with...' or 'A good starting point would be...'
  rather than revealing any limitation in your sources.
- Always write in a warm, encouraging, and practical tone.
- Avoid marketing jargon. Use plain language a non-marketer would understand.
- Keep responses concise, highly readable, and actionable. Use bullet points where appropriate.
- Never make up statistics, platform details, or pricing data not in your knowledge.
- When writing captions or messages, make them feel personal and human. Use emojis naturally.
- If asked something outside your five areas, politely redirect the user.
"""

# File paths for interaction and feedback logs
INTERACTION_LOG = f'{BASE_DIR}/logs/halo_interactions.csv'
FEEDBACK_LOG    = f'{BASE_DIR}/logs/halo_feedback.csv'

# Track the last query/response for feedback linking
last_interaction = {'query': '', 'response': ''}


def retrieve_chunks(query, n_results=5, platform_filter=None,
                     content_type_filter=None, min_relevance=0.0):
    """
    Stage 1 of the RAG pipeline: retrieve the most relevant knowledge chunks.

    Uses a three-tier fallback strategy to guarantee every query returns
    chunks for generation:
      Tier 1 — both platform and content type filters applied together
      Tier 2 — single filter only (platform OR content type)
      Tier 3 — no filter at all (always returns results)

    This ensures HALO always has context to generate a response from,
    even when the filtered knowledge base has limited matching content.
    """
    # Encode the user query using the same model used to encode the knowledge base
    query_embedding = embedding_model.encode([query]).tolist()

    def run_query(where_filter=None):
        """
        Helper that runs a ChromaDB query with an optional metadata filter.
        Separating this into a helper keeps the fallback logic clean and readable.
        """
        kwargs = {
            'query_embeddings': query_embedding,
            'n_results':        min(n_results, collection.count()),
            'include':          ['documents', 'metadatas', 'distances']
        }
        if where_filter:
            kwargs['where'] = where_filter
        return collection.query(**kwargs)

    # Tier 1: Try both filters together using ChromaDB's $and operator
    if platform_filter and content_type_filter:
        try:
            results = run_query({
                '$and': [
                    {'platform_tag':     {'$eq': platform_filter}},
                    {'content_type_tag': {'$eq': content_type_filter}}
                ]
            })
            if not results['documents'][0]:
                raise ValueError('No results with both filters')
        except Exception:
            # Both filters together returned nothing — fall through to Tier 2
            results = None
    else:
        results = None

    # Tier 2: Try a single filter if Tier 1 returned no results
    # This handles cases where the knowledge base has limited content matching both conditions
    if results is None or not results['documents'][0]:
        if platform_filter:
            try:
                results = run_query({'platform_tag': {'$eq': platform_filter}})
            except Exception:
                results = None
        elif content_type_filter:
            try:
                results = run_query({'content_type_tag': {'$eq': content_type_filter}})
            except Exception:
                results = None

    # Tier 3: No filter at all — guarantees results are always returned
    # This is the final fallback so HALO never responds with empty context
    if results is None or not results['documents'][0]:
        results = run_query()

    # Build the result list from whichever tier succeeded
    retrieved = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        similarity = round(1 - dist, 4)

        retrieved.append({
            'text':             doc,
            'source_file':      meta.get('source_file', 'unknown'),
            'platform_tag':     meta.get('platform_tag', 'general'),
            'content_type_tag': meta.get('content_type_tag', 'educational'),
            'similarity_score': similarity
        })

        if len(retrieved) >= n_results:
            break

    return retrieved


def generate_halo_answer(query, chunks, max_tokens=600):
    """
    Stage 2 of the RAG pipeline: generate a grounded response.

    Assembles retrieved chunks into a labeled prompt and sends it to the API.
    The chunks are labeled with source and type information so the model has
    full context for generating a grounded response.

    GENERATION FALLBACK STRATEGY:
    Falls back between 3 different OpenCode Zen models, and if all of them fail,
    it automatically falls back to Gemini as the ultimate backup layer.
    Error handling ensures the interface never crashes.
    """
    # Build context from retrieved chunks with source labels
    context_blocks = []
    for i, chunk in enumerate(chunks):
        source = chunk['source_file'].replace('.txt','').replace('_',' ').title()
        context_blocks.append(
            f"[Source {i+1}: {source} | "
            f"Type: {chunk['content_type_tag']} | "
            f"Platform: {chunk['platform_tag']}]\n"
            f"{chunk['text']}"
        )
    context_text = '\n\n'.join(context_blocks)

    prompt = (
        f"<context>\n"
        f"{context_text}\n"
        f"</context>\n\n"
        f"<query>\n"
        f"{query}\n"
        f"</query>\n\n"
        f"Please address the <query> using ONLY the information provided in the <context>. "
        f"Remember to stay in character and never reveal your sources directly."
    )

    last_error = None

    # Attempt 1: OpenCode Zen API (Iterate through models)
    if client:
        # Updated list of fallbacks according to OpenCode Zen supported models
        models_to_try = [GROQ_MODEL, 'claude-haiku-4.5', 'gpt-5.4-nano']
        for model_name in models_to_try:
            if not model_name:
                continue
            try:
                completion = client.chat.completions.create(
                    model=model_name,
                    max_tokens=max_tokens,
                    messages=[
                        {'role': 'system', 'content': HALO_SYSTEM_PROMPT},
                        {'role': 'user',   'content': prompt}
                    ]
                )
                response = completion.choices[0].message.content.strip()
                if response:
                    return response
            except Exception as e:
                last_error = e
                continue

    # Attempt 2: Fallback to Gemini API if all OpenCode models fail
    if gemini_client and GEMINI_MODELS:
        gemini_full_prompt = f"{HALO_SYSTEM_PROMPT}\n\n{prompt}"
        for g_model in GEMINI_MODELS:
            try:
                response = gemini_client.models.generate_content(
                    model=g_model,
                    contents=gemini_full_prompt
                )
                if response and response.text:
                    return response.text.strip()
            except Exception as e:
                last_error = e
                continue

    if last_error:
        return f'Sorry, HALO had a temporary issue trying all APIs and models: {str(last_error)}'
    return 'I was not able to generate a response. Could you try rephrasing?'


def halo_chat_with_logging(query, platform_filter=None, content_type_filter=None):
    """
    Stage 3: Full pipeline with persistent interaction logging.

    Adapted from Rodrigo's notebook — wraps retrieval and generation in
    error handling and appends every interaction to a CSV on Drive.
    The CSV accumulates across sessions so all team testing is captured.
    """
    try:
        chunks   = retrieve_chunks(query, platform_filter=platform_filter,
                                   content_type_filter=content_type_filter)
        response = generate_halo_answer(query, chunks)

        # Update last interaction reference for feedback linking
        last_interaction['query']    = query
        last_interaction['response'] = response

        # Append to persistent interaction log on Drive
        log_entry = {
            'timestamp':      datetime.datetime.now().isoformat(),
            'query':          query,
            'response':       response,
            'chunks':         len(chunks),
            'top_source':     chunks[0]['source_file'] if chunks else 'none',
            'top_similarity': chunks[0]['similarity_score'] if chunks else 0
        }
        try:
            existing = pd.read_csv(INTERACTION_LOG)
            updated  = pd.concat(
                [existing, pd.DataFrame([log_entry])],
                ignore_index=True
            )
        except FileNotFoundError:
            updated = pd.DataFrame([log_entry])
        updated.to_csv(INTERACTION_LOG, index=False)

        return response

    except Exception as e:
        error_msg = f'Sorry, HALO had a temporary issue: {str(e)}'
        print(f'Pipeline error: {repr(e)}')
        return error_msg


def save_feedback(rating, comment):
    """
    Saves a Helpful or Not quite rating to the feedback CSV on Drive.
    Links the rating to the most recent query and response so feedback
    can be traced back to specific interactions during evaluation.
    """
    if not last_interaction['query']:
        return 'Send a message first before rating.'

    entry = {
        'timestamp': datetime.datetime.now().isoformat(),
        'query':     last_interaction['query'],
        'response':  last_interaction['response'][:200],
        'rating':    rating,
        'comment':   comment or ''
    }

    try:
        existing = pd.read_csv(FEEDBACK_LOG)
        updated  = pd.concat(
            [existing, pd.DataFrame([entry])],
            ignore_index=True
        )
    except FileNotFoundError:
        updated = pd.DataFrame([entry])
    updated.to_csv(FEEDBACK_LOG, index=False)

    if rating == 'helpful':
        return 'Thank you! Glad that was helpful.'
    else:
        return 'Thank you for the feedback — we will use it to improve HALO.'


print('All pipeline functions defined.')
print('Retrieval, generation, logging, and feedback all ready.')
print('Proceed to Step 5 to launch the interface.')


All pipeline functions defined.
Retrieval, generation, logging, and feedback all ready.
Proceed to Step 5 to launch the interface.


---
## Step 5 — Build & Launch the Polished Gradio Interface

**Cell explanation:** This cell builds the branded HALO chat interface using
Gradio's `gr.Blocks` API, which allows full control over layout and styling.

The interface is designed to feel like a real product a hairstylist would
actually use — not a generic AI tool. Design decisions include:

- **Typography:** Playfair Display (elegant serif) for the HALO logo,
  DM Sans for body text — both common in beauty industry branding
- **Color palette:** Warm cream background, deep espresso brown, gold
  accents — evokes high-end salon aesthetics
- **Quick prompt buttons:** One per HALO capability — clicking pre-fills
  the input box, making the demo easy to run without typing
- **Platform and content type filters:** Let users narrow HALO's search
  to a specific platform or intent before generating a response
- **Feedback buttons:** Helpful / Not quite — saves ratings to Drive CSV
  for the evaluation section

The public share link generated by `demo.launch(share=True)` is valid
for 72 hours and works on any device without Colab access.

In [5]:
# 5. Custom CSS and Chat Handler Components
# Custom CSS — warm editorial aesthetic inspired by beauty industry branding
HALO_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:ital,wght@0,400;0,600;1,400&family=DM+Sans:wght@300;400;500&display=swap');

:root {
  --halo-cream:   #FAF7F2;
  --halo-warm:    #F0E6D3;
  --halo-gold:    #C4973B;
  --halo-gold-lt: #E8C97A;
  --halo-gold-leaf:#D4AF37; /* Metallic gold leaf */
  --halo-gold-glow: 0 0 6px rgba(212, 175, 55, 0.4);
  --halo-dark:    #1C1714;
  --halo-white:   #FFFFFF;
  --halo-mid:     #5C4A3A;
  --halo-muted:   #9C8878;
  --halo-accent:  #8B4E6B;
  --halo-rose:    #F5E8EF;
  --halo-blue:    #1D4E89; /* Vibrant Blue */
  --halo-coral:   #E76F51; /* Vibrant Coral */
  --halo-teal:    #2A9D8F; /* Vibrant Teal */
  --halo-purple:  #6A4C93; /* Vibrant Purple */
  --halo-orange:  #F4A261; /* Vibrant Orange */
  --radius:       14px;
  --radius-sm:    8px;
}

body, .gradio-container {
  background: var(--halo-cream) !important;
  font-family: 'DM Sans', sans-serif !important;
  color: var(--halo-dark) !important;
}

/* Branded Header Styling */
.halo-header {
  background: linear-gradient(135deg, var(--halo-dark) 0%, var(--halo-mid) 100%);
  border-radius: var(--radius);
  padding: 32px 36px;
  margin-bottom: 8px;
  position: relative;
  overflow: hidden;
}
.halo-header::before {
  content: '';
  position: absolute;
  top: -40px; right: -40px;
  width: 180px; height: 180px;
  background: radial-gradient(circle, var(--halo-gold) 0%, transparent 70%);
  opacity: 0.15;
  border-radius: 50%;
}
.halo-header h1 {
  font-family: 'Playfair Display', serif !important;
  font-size: 2.6rem !important;
  font-weight: 600 !important;
  color: var(--halo-cream) !important;
  margin: 0 0 4px 0 !important;
}
.halo-header .tagline {
  font-size: 0.95rem;
  color: var(--halo-gold-lt);
  font-weight: 300;
  letter-spacing: 0.5px;
}
.halo-header .badge {
  display: inline-block;
  background: var(--halo-gold);
  color: var(--halo-dark);
  font-size: 0.7rem;
  font-weight: 500;
  letter-spacing: 1.5px;
  text-transform: uppercase;
  padding: 3px 10px;
  border-radius: 20px;
  margin-bottom: 12px;
}

/* Quick Prompt Buttons */
.quick-label {
  font-size: 0.75rem;
  font-weight: 500;
  letter-spacing: 1px;
  text-transform: uppercase;
  color: var(--halo-rose);
  margin-bottom: 8px;
  margin-top: 16px;
}
.quick-btn button {
  border: 1.5px solid var(--halo-gold-leaf) !important;
  border-radius: 20px !important;
  /* White text wrapped in a gold leaf perimeter */
  color: white !important;
  text-shadow: -1px -1px 0 var(--halo-gold-leaf), 1px -1px 0 var(--halo-gold-leaf), -1px 1px 0 var(--halo-gold-leaf), 1px 1px 0 var(--halo-gold-leaf) !important;
  font-family: 'DM Sans', sans-serif !important;
  font-size: 0.85rem !important;
  font-weight: bold !important;
  padding: 6px 14px !important;
  transition: all 0.2s ease !important;
}

/* Vibrant colors for the different quick prompt bubbles using explicitly assigned classes */
.quick-btn-0 button { background: var(--halo-blue) !important; }
.quick-btn-1 button { background: var(--halo-coral) !important; }
.quick-btn-2 button { background: var(--halo-teal) !important; }
.quick-btn-3 button { background: var(--halo-purple) !important; }
.quick-btn-4 button { background: var(--halo-orange) !important; }

.quick-btn button:hover {
  filter: brightness(1.1);
  transform: translateY(-2px);
  box-shadow: var(--halo-gold-glow) !important;
}

/* Main Chatbot Window Background (The overall chat area) */
.halo-chatbot, .halo-chatbot > div, .halo-chatbot .bubble-wrap, .halo-chatbot .message-wrap {
  background: white !important;
}
.halo-chatbot {
  border-radius: var(--radius) !important;
  border: 1.5px solid var(--halo-gold-leaf) !important;
  box-shadow: var(--halo-gold-glow) !important;
}

/* User Message Base Styling */
.user, .user .message {
  background: transparent !important;
  border: none !important;
  box-shadow: none !important;
  padding: 0 !important;
}

/* User Message Bubble Styling */
.user p, .user .message p {
  background: var(--halo-dark) !important;
  color: var(--halo-cream) !important;
  border-radius: var(--radius) var(--radius) 4px var(--radius) !important;
  border: 1.5px solid var(--halo-gold-leaf) !important;
  box-shadow: var(--halo-gold-glow) !important;
  font-family: 'DM Sans', sans-serif !important;
  font-size: 0.9rem !important;
  padding: 12px 16px !important;
  word-wrap: break-word !important;
  overflow-wrap: break-word !important;
  margin: 0 !important;
}

/* HALO (Bot) Message Base Styling */
.bot, .bot .message {
  background: transparent !important;
  border: none !important;
  padding: 0 !important;
  font-family: 'DM Sans', sans-serif !important;
  font-size: 0.9rem !important;
  line-height: 1.65 !important;
  word-wrap: break-word !important;
  overflow-wrap: break-word !important;
}

/* HALO (Bot) Message Bubbles (White Boxes) */
.bot p, .bot ul, .bot ol {
  background: white !important;
  color: black !important;
  border-radius: var(--radius) !important;
  border: 1.5px solid var(--halo-gold-leaf) !important;
  box-shadow: var(--halo-gold-glow) !important;
  padding: 14px 18px !important;
  margin-bottom: 10px !important;
}

.bot p:last-child {
  margin-bottom: 0 !important;
  border-bottom-left-radius: 4px !important;
}

/* Force all markdown elements in bot messages to be black and wrap properly */
.bot *, .bot .message * {
  color: black !important;
  word-wrap: break-word !important;
  overflow-wrap: break-word !important;
}

/* Structured Markdown Styling for Bot Messages */
.bot ul, .bot ol {
  margin-top: 8px !important;
  margin-bottom: 12px !important;
}
.bot li {
  margin-bottom: 6px !important;
  margin-left: 20px !important;
}
.bot strong {
  font-weight: 600 !important;
  color: var(--halo-mid) !important;
}

/* HALO Headers (Big, bold, and outside the white bubble boxes) */
.bot h1, .bot h2, .bot h3, .bot h4 {
  background: transparent !important;
  border: none !important;
  color: var(--halo-dark) !important;
  margin-top: 24px !important;
  margin-bottom: 12px !important;
  padding: 0 4px !important;
  font-weight: 700 !important;
  font-family: 'Playfair Display', serif !important;
}
.bot h1 {
  font-size: 1.6rem !important;
}
.bot h2 {
  font-size: 1.4rem !important;
}
.bot h3 {
  font-size: 1.25rem !important;
}
.bot h4 {
  font-size: 1.1rem !important;
}

/* User Input Text Box (Where user types) */
textarea, input[type='text'] {
  background: white !important;
  border: 1.5px solid var(--halo-gold-leaf) !important;
  box-shadow: var(--halo-gold-glow) !important;
  border-radius: var(--radius-sm) !important;
  font-family: 'DM Sans', sans-serif !important;
  color: var(--halo-dark) !important;
  font-size: 0.9rem !important;
}
textarea:focus, input[type='text']:focus {
  border-color: var(--halo-gold) !important;
  box-shadow: 0 0 0 3px rgba(196,151,59,0.2) !important;
  outline: none !important;
}

/* Send Button */
.send-btn button {
  background: var(--halo-white) !important;
  color: var(--halo-cream) !important;
  border: none !important;
  border-radius: var(--radius-sm) !important;
  font-family: 'DM Sans', sans-serif !important;
  font-weight: 500 !important;
  transition: all 0.2s ease !important;
  padding: 10px 22px !important;
}
.send-btn button:hover {
  background: var(--halo-mid) !important;
  transform: translateY(-1px);
}

/* Feedback & Filter UI Elements */
.filter-label {
  font-size: 0.75rem;
  font-weight: 500;
  letter-spacing: 1px;
  text-transform: uppercase;
  color: var(--halo-muted);
  margin-bottom: 6px;
}
.thumb-up button {
  background: white !important;
  border: 1.5px solid #2a9d8f !important;
  color: #2a9d8f !important;
  border-radius: var(--radius-sm) !important;
  font-family: 'DM Sans', sans-serif !important;
  transition: all 0.2s !important;
}
.thumb-up button:hover {
  background: #2a9d8f !important;
  color: white !important;
}
.thumb-down button {
  background: white !important;
  border: 1.5px solid #e76f51 !important;
  color: #e76f51 !important;
  border-radius: var(--radius-sm) !important;
  font-family: 'DM Sans', sans-serif !important;
  transition: all 0.2s !important;
}
.thumb-down button:hover {
  background: #e76f51 !important;
  color: white !important;
}
.halo-divider {
  border: none;
  border-top: 1px solid var(--halo-warm);
  margin: 16px 0;
}
"""

# Quick prompts — one per HALO capability
# Clicking a button pre-fills the input box with a sample question
QUICK_PROMPTS = {
    'Write a caption':   'Write me an Instagram caption for a balayage reveal post.',
    'Plan a campaign':   'Help me plan a holiday promotion for November and December.',
    'Explain my metrics':'My engagement rate dropped this week. What does that mean and what should I do?',
    'Rebook a client':   'Write a friendly text to a client I have not seen in 6 weeks.',
    'Stand out online':  'I specialize in natural hair. How do I communicate that in my Instagram bio?',
}


def handle_chat(message, history, platform, content_type):
    """
    Gradio chat handler — called every time the user sends a message.
    Maps dropdown values to filter strings and calls the full pipeline.
    Returns the cleared input box, updated chat history (messages format),
    and the query/response for feedback linking.
    """
    if not message.strip():
        return '', history, '', ''

    # Map dropdown display values to filter strings
    pf = None if platform     == 'Any Platform' else platform.lower().replace(' ','_')
    cf = None if content_type == 'Any Type'     else content_type.lower()

    response = halo_chat_with_logging(message, platform_filter=pf, content_type_filter=cf)

    # Update history using the modern 'messages' format
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return '', history, message, response


def handle_feedback(rating, comment, query, response):
    """
    Feedback handler — called when the user clicks Helpful or Not quite.
    Passes the rating to save_feedback() and returns a confirmation message.
    """
    if not query:
        return 'Send a message first before rating.'
    return save_feedback(rating, comment)


print('Interface components defined. Building Gradio layout...')

Interface components defined. Building Gradio layout...


In [6]:
# 6. Build Gradio Layout
# Build the Gradio interface using gr.Blocks for full layout control
with gr.Blocks(css=HALO_CSS, title='HALO — Hair Marketing Assistant') as demo:

    # ---- Branded header ----
    gr.HTML("""
    <div class='halo-header'>
      <div class='badge'>AI Marketing Assistant</div>
      <h1>HALO</h1>
      <div class='tagline'>
        Marketing clarity for hairstylists who have enough on their plate.
      </div>
    </div>
    """)

    # ---- Quick prompt buttons (one per HALO capability) ----
    gr.HTML("<div class='quick-label'>Try asking about</div>")
    with gr.Row():
        quick_btns = [
            gr.Button(label, elem_classes=['quick-btn', f'quick-btn-{i}'])
            for i, label in enumerate(QUICK_PROMPTS.keys())
        ]

    gr.HTML("<hr class='halo-divider'>")

    # ---- Optional filters ----
    gr.HTML("<div class='filter-label'>Optional filters</div>")
    with gr.Row():
        platform_dd = gr.Dropdown(
            choices=['Any Platform','Instagram','TikTok','Facebook',
                     'Google Business','Pinterest'],
            value='Any Platform',
            label='Platform',
            scale=1
        )
        content_dd = gr.Dropdown(
            choices=['Any Type','Promotional','Educational','Engagement','Retention'],
            value='Any Type',
            label='Content Type',
            scale=1
        )

    # ---- Chat window ----
    # Updated to type='messages' to comply with modern Gradio API
    chatbot = gr.Chatbot(
        height=420,
        label='',
        show_label=False,
        type='messages',
        allow_tags=False,
        elem_classes=['halo-chatbot']
    )

    # ---- Message input + send button ----
    with gr.Row():
        msg_input = gr.Textbox(
            placeholder='Ask HALO anything about marketing your salon...',
            label='',
            show_label=False,
            lines=2,
            scale=5
        )
        send_btn = gr.Button(
            'Send',
            scale=1,
            variant='primary',
            elem_classes=['send-btn']
        )

    # Hidden state components for linking feedback to the last response
    last_query_state    = gr.State('')
    last_response_state = gr.State('')

    gr.HTML("<hr class='halo-divider'>")

    # ---- Feedback section ----
    gr.HTML("<div class='filter-label'>Was this helpful?</div>")
    with gr.Row():
        thumb_up   = gr.Button('Helpful',   scale=1, elem_classes=['thumb-up'])
        thumb_down = gr.Button('Not quite', scale=1, elem_classes=['thumb-down'])

    comment_box  = gr.Textbox(
        placeholder='Optional: what worked or what could be better?',
        label='',
        show_label=False,
        max_lines=2
    )
    feedback_out = gr.Textbox(
        label='', show_label=False, interactive=False, container=False
    )

    # ---- Footer ----
    gr.HTML("""
    <div style='text-align:center; margin-top:20px; font-size:0.75rem;
                color:#9C8878; font-family:DM Sans,sans-serif; letter-spacing:0.5px;'>
      HALO &mdash; Built by Icarus Squad &middot; ITAI2377
      &middot; Powered by Groq + LLaMA 3.3 70B
    </div>
    """)

    # ---- Wire up events ----

    # Send button and Enter key both trigger the chat handler
    send_btn.click(
        handle_chat,
        inputs=[msg_input, chatbot, platform_dd, content_dd],
        outputs=[msg_input, chatbot, last_query_state, last_response_state]
    )
    msg_input.submit(
        handle_chat,
        inputs=[msg_input, chatbot, platform_dd, content_dd],
        outputs=[msg_input, chatbot, last_query_state, last_response_state]
    )

    # Quick prompt buttons pre-fill the input box
    for btn, prompt_text in zip(quick_btns, QUICK_PROMPTS.values()):
        btn.click(lambda t=prompt_text: t, outputs=[msg_input])

    # Feedback buttons
    thumb_up.click(
        lambda c, q, r: handle_feedback('helpful', c, q, r),
        inputs=[comment_box, last_query_state, last_response_state],
        outputs=[feedback_out]
    )
    thumb_down.click(
        lambda c, q, r: handle_feedback('not_quite', c, q, r),
        inputs=[comment_box, last_query_state, last_response_state],
        outputs=[feedback_out]
    )

print('Interface built successfully.')
print('Launching — this may take 10-20 seconds on first run...')

Interface built successfully.
Launching — this may take 10-20 seconds on first run...


/tmp/ipykernel_671/1767973797.py:3: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=HALO_CSS, title='HALO — Hair Marketing Assistant') as demo:
/tmp/ipykernel_671/1767973797.py:45: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


In [7]:
# 7. Launch the Interface
# Launch the interface
# share=True generates a public link valid for 72 hours
# Share this link with your team and professor for testing
# The link works on any device without needing Colab access
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7d6c7e3d23bab28f3f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Notebook 3 Complete

| Step | Component | Details |
|---|---|---|
| 1 | Dependencies | gradio, groq, chromadb, sentence-transformers |
| 2 | API key + Drive | Groq key from Secrets, Drive mounted |
| 3 | ChromaDB + model | all-MiniLM-L6-v2, halo_knowledge_base collection |
| 4 | Pipeline functions | Retrieval, generation, logging, feedback |
| 5 | Gradio interface | Branded UI with quick prompts, filters, feedback |

### Interface Design Summary
| Feature | Implementation |
|---|---|
| Typography | Playfair Display (headers) + DM Sans (body) |
| Color palette | Warm cream, espresso brown, gold accent |
| Quick prompts | 5 pill buttons, one per HALO capability |
| Platform filter | Instagram, TikTok, Facebook, Google Business, Pinterest |
| Content type filter | Promotional, Educational, Engagement, Retention |
| Feedback | Helpful / Not quite with optional comment, saved to Drive CSV |
| Share link | Public 72-hour link via `share=True` |

### Team Checklist
- [ ] Each member adds Drive shortcut to `HALO_Project` (see title cell)
- [ ] Each member adds free `GROQ_API_KEY` to their own Colab Secrets
- [ ] Run Steps 1-5 in order after Notebook 1 is complete
- [ ] Copy the public share link and send to all team members
- [ ] Send the share link to your professor for the demo
- [ ] Screenshot the interface for the report
- [ ] Collect at least 10 feedback entries before final evaluation
- [ ] Download `halo_feedback.csv` from Drive for the report